# 钢板缺陷分类（Faulty Steel Plates）

数据来源：https://www.kaggle.com/datasets/uciml/faulty-steel-plates （UCI 数据集）

数据集包含 1941 条钢板样本：

- **前 27 列**：从缺陷图像中提取的几何/光度特征（位置、面积、周长、亮度统计量等）
- **后 7 列**：缺陷类别，以独热编码存储（每行恰好一个 1，类别互斥）

## 任务定位：有监督学习

每条样本都带有已知标签（7 种缺陷类型之一），训练时模型对照标签学习
“特征 → 类别”的映射，并在没见过的测试集上评估泛化能力。
属于**多分类（Multiclass Classification）**问题。

## 流程

1. 数据加载与标签还原（独热 → 单标签列）
2. KNN 多分类（基线模型）
3. SVM 多分类（线性核 vs RBF 核）
4. GridSearchCV 调参


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_blobs
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

In [ ]:
csv_path = Path.cwd() / "faults.csv"

if not csv_path.exists():
    csv_path = Path("Python Guidance/day46-60/faults.csv")

df = pd.read_csv(csv_path)
df.head()

In [ ]:
FEATURES = df.columns[:27]  # 前 27 个特征
LABELS = df.columns[27:]    # 后 7 个类别列（独热编码）

# 独热 -> 单一标签列：取每行值为 1 的那一列的列名
y = df[LABELS].idxmax(axis=1)
X = df[FEATURES]

# 类别分布很不均衡，先看一下
y.value_counts()


说明：

- 7 个类别互斥，属于**单标签多分类**，用 `idxmax` 把独热编码还原成
  单个标签列即可，无需多标签算法（Binary Relevance 等）。
- 从 `value_counts` 可见类别很不均衡：`Other_Faults` 有 673 条，
  而 `Dirtiness` 只有 55 条。因此划分数据集时要 `stratify=y`（分层抽样），
  评估时也要关注 macro 指标而不是只看 accuracy。


## KNN 多分类

7 个类别互斥（每行恰好一个 1），属于单标签多分类问题，
用 `idxmax` 把独热编码还原成单个标签列即可直接训练 KNN。
KNN 对特征尺度敏感，务必用 Pipeline 做标准化，避免数据泄漏。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y  # 分层抽样，保持各类比例
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5, weights="distance")),
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
ConfusionMatrixDisplay.from_estimator(
    model, X_test, y_test, xticks_rotation=45, cmap="Blues"
)
plt.tight_layout()
plt.show()


## SVM支持向量机

In [ ]:
# 线性核 SVM，class_weight="balanced" 处理类别不均衡
svm_linear = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1.0, class_weight="balanced", random_state=42)),
])
svm_linear.fit(X_train, y_train)

y_pred_svm_linear = svm_linear.predict(X_test)
print(classification_report(y_test, y_pred_svm_linear))


SVM 本质上是二分类器，scikit-learn 的 `SVC` 默认用 One-vs-Rest (OVR)
策略处理多分类：为 7 个类别各训练一个“是/否该类”的 SVM。
与 KNN 一样，SVM 对特征尺度敏感，同样需要标准化。
上面已用线性核训练了一个 SVM；下面再试默认的高斯 RBF 核，
1941 条样本规模不大，`SVC` 可以直接用（不需要 `LinearSVC` 那种近似）。


In [ ]:
# 高斯 RBF 核（SVC 的默认核），拟合能力通常比线性核更强
svm_rbf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(class_weight="balanced", random_state=42)),
])
svm_rbf.fit(X_train, y_train)

y_pred_svm_rbf = svm_rbf.predict(X_test)
print(classification_report(y_test, y_pred_svm_rbf))


In [ ]:
ConfusionMatrixDisplay.from_estimator(
    svm_rbf, X_test, y_test, xticks_rotation=45, cmap="Blues"
)
plt.tight_layout()
plt.show()


In [ ]:
# 三种模型的 macro F1 对比（macro 对少数类更敏感，适合不均衡数据）
print(f"KNN        macro F1: {f1_score(y_test, y_pred, average='macro'):.3f}")
print(f"SVM linear macro F1: {f1_score(y_test, y_pred_svm_linear, average='macro'):.3f}")
print(f"SVM rbf    macro F1: {f1_score(y_test, y_pred_svm_rbf, average='macro'):.3f}")


## 网格搜索（GridSearchCV）

对 RBF 核 SVM 的两个关键超参数做网格搜索：

- `C`：正则化强度的倒数，越大拟合越激进
- `gamma`：RBF 核的宽度，越大决策边界越曲折

注意点：
1. 搜索要落在 **Pipeline** 上进行，保证每折 CV 都只用训练部分拟合 scaler，不泄漏。
2. 多分类任务评分指标用 `f1_macro`（普通的 `f1` 只适用于二分类）。
3. 参数名要写 Pipeline 中的步骤名前缀，如 `svm__C`。


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold


In [ ]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(class_weight="balanced", random_state=42)),
])

# 分层 K 折，保持每折的类别比例（数据不均衡时尤其重要）
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


In [ ]:
param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", "auto", 0.01, 0.1],
}

grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",   # 多分类 + 不均衡数据，用 macro F1
    cv=cv,
    n_jobs=-1,
    refit=True,           # 用最佳参数在整个训练集上重新拟合
)
grid_search.fit(X_train, y_train)


In [ ]:
print("Grid Search 最佳参数:")
print(grid_search.best_params_)
print(f"\nGrid Search 最佳交叉验证 macro F1: {grid_search.best_score_:.4f}")


## Random Search随机搜索

In [ ]:
param_distributions = {
    "svm__C": [0.1, 0.3, 1, 3, 10, 30,  100],
    "svm__gamma": ["scale", "auto", 0.01, 0.1, 1.0],
}

random_search = RandomizedSearchCV(
    estimator=svm_pipeline,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="f1_macro",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=0,
)

random_search.fit(X_train, y_train)

In [ ]:
print("Random Search 最佳参数:")
print(random_search.best_params_)
print(f"\nRandom Search 最佳交叉验证 macro F1: {random_search.best_score_:.4f}")

In [ ]:
# 用调参后的最佳模型评估测试集
y_pred_svm_tuned = grid_search.predict(X_test)
print(classification_report(y_test, y_pred_svm_tuned))


## 小结

| 模型 | macro F1 |
|---|---|
| KNN (k=5, distance) | 0.781 |
| SVM RBF（默认参数） | 0.754 |
| SVM RBF（GridSearchCV 调参后） | 0.773 |

要点回顾：

1. 独热标签还原成单标签列后，多分类可以直接用 KNN / SVC 原生支持。
2. KNN 和 SVM 都对特征尺度敏感，必须用 Pipeline 标准化，防止数据泄漏。
3. 类别不均衡时：`stratify` 分层抽样、`class_weight="balanced"`、
   macro 指标三者缺一不可。
4. 网格搜索要落在 Pipeline 上、用 `f1_macro` 评分、参数名带 `步骤名__` 前缀。


In [ ]:
# 随机搜索的最佳模型在测试集上的表现
y_pred_svm_rand = random_search.predict(X_test)
print(classification_report(y_test, y_pred_svm_rand))

print(f"Grid   Search 测试集 macro F1: {f1_score(y_test, y_pred_svm_tuned, average='macro'):.3f}")
print(f"Random Search 测试集 macro F1: {f1_score(y_test, y_pred_svm_rand, average='macro'):.3f}")
